In [3]:
pip install gymnasium


   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   ---------------------------------------- 2/2 [gymnasium]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\zha02\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
import gymnasium as gym
print(gym.__version__)

1.4.0


In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)
print(env)

<TimeLimit<OrderEnforcing<PassiveEnvChecker<FrozenLakeEnv<FrozenLake-v1>>>>>


In [3]:
state, info = env.reset()
print("State:", state)
print("Info:", info)

State: 0
Info: {'prob': 1}


In [4]:
action = 2  # Right

next_state, reward, terminated, truncated, info = env.step(action)

print("Next state:", next_state)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Info:", info)

Next state: 1
Reward: 0
Terminated: False
Truncated: False
Info: {'prob': 1.0}


In [5]:
state, info = env.reset()

done = False

while not done:
    action = env.action_space.sample()

    state, reward, terminated, truncated, info = env.step(action)

    print(
        "State:", state,
        "Reward:", reward,
        "Terminated:", terminated,
        "Truncated:", truncated
    )

    done = terminated or truncated

State: 0 Reward: 0 Terminated: False Truncated: False
State: 1 Reward: 0 Terminated: False Truncated: False
State: 5 Reward: 0 Terminated: True Truncated: False


In [6]:
state, info = env.reset()

total_reward = 0
steps = 0

while steps < 100:
    action = env.action_space.sample()

    state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps += 1

    if terminated or truncated:
        break

print("Steps:", steps)
print("Total reward:", total_reward)
print("Final state:", state)
print("Terminated:", terminated)
print("Truncated:", truncated)

Steps: 5
Total reward: 0
Final state: 5
Terminated: True
Truncated: False


In [7]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="ansi"
)

state, info = env.reset()

print(env.render())


SFFF
FHFH
FFFH
HFFG



In [8]:
import os
os.makedirs("outputs", exist_ok=True)
print("Folder created")

Folder created


In [9]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="ansi"
)

state, info = env.reset()

frames = []
frames.append(str(env.render()))

done = False
steps = 0

while not done and steps < 100:
    action = env.action_space.sample()

    state, reward, terminated, truncated, info = env.step(action)

    frames.append(str(env.render()))

    done = terminated or truncated
    steps += 1

with open("outputs/episode_trace.txt", "w", encoding="utf-8") as f:
    for i, frame in enumerate(frames):
        f.write(f"Step {i}\n")
        f.write(frame)
        f.write("\n\n")

print("Saved.")

Saved.


In [10]:
for s in range(16):
    print(s)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15


In [12]:
fixed_policy = {
    0: 2,   # Right
    1: 2,   # Right
    2: 1,   # Down
    3: 0,   # Left

    4: 0,   # Left
    6: 1,   # Down

    8: 3,   # Up
    9: 2,   # Right
    10: 1,  # Down

    13: 2,  # Right
    14: 2   # Right -> Goal
}
for state, action in fixed_policy.items():
    print(f"State {state} -> Action {action}")

State 0 -> Action 2
State 1 -> Action 2
State 2 -> Action 1
State 3 -> Action 0
State 4 -> Action 0
State 6 -> Action 1
State 8 -> Action 3
State 9 -> Action 2
State 10 -> Action 1
State 13 -> Action 2
State 14 -> Action 2


In [13]:
state, info = env.reset()

steps = 0
total_reward = 0

while steps < 100:
    
    action = fixed_policy.get(state, 0)

    state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps += 1

    print(
        f"Step {steps}:",
        f"state={state}",
        f"reward={reward}"
    )

    if terminated or truncated:
        break

print("\nFinal state:", state)
print("Steps:", steps)
print("Total reward:", total_reward)

Step 1: state=1 reward=0
Step 2: state=2 reward=0
Step 3: state=6 reward=0
Step 4: state=10 reward=0
Step 5: state=14 reward=0
Step 6: state=15 reward=1

Final state: 15
Steps: 6
Total reward: 1


In [14]:
def run_episode(env, policy="random"):
    
    state, info = env.reset()

    total_reward = 0
    steps = 0

    while steps < 100:

        if policy == "random":
            action = env.action_space.sample()
        else:
            action = fixed_policy.get(state, 0)

        state, reward, terminated, truncated, info = env.step(action)

        total_reward += reward
        steps += 1

        if terminated or truncated:
            break

    return state, total_reward, steps, terminated

In [15]:
state, total_reward, steps, terminated = run_episode(
    env,
    policy="random"
)

print("State:", state)
print("Reward:", total_reward)
print("Steps:", steps)
print("Terminated:", terminated)

State: 12
Reward: 0
Steps: 10
Terminated: True


In [16]:
import random
import pandas as pd

In [17]:
results = []

for i in range(50):

    state, total_reward, steps, terminated = run_episode(
        env,
        policy="random"
    )

    results.append([
        i,
        state,
        total_reward,
        steps,
        terminated
    ])

print("Episodes:", len(results))

Episodes: 50


In [18]:
def get_outcome(state, reward):

    if reward == 1:
        return "goal"

    elif state in [5, 7, 11, 12]:
        return "hole"

    else:
        return "time_limit"

In [19]:
def run_episode(env, policy="random", seed=None):

    state, info = env.reset(seed=seed)

    total_reward = 0
    steps = 0

    while steps < 100:

        if policy == "random":
            action = env.action_space.sample()
        else:
            action = fixed_policy.get(state, 0)

        state, reward, terminated, truncated, info = env.step(action)

        total_reward += reward
        steps += 1

        if terminated or truncated:
            break

    outcome = get_outcome(state, total_reward)

    return state, total_reward, steps, outcome

In [20]:
batch_seed = 11

rows = []

for i in range(50):

    episode_seed = 1000 * batch_seed + i

    state, total_reward, steps, outcome = run_episode(
        env,
        policy="random",
        seed=episode_seed
    )

    rows.append([
        batch_seed,
        episode_seed,
        "random",
        steps,
        total_reward,
        outcome
    ])

print("Rows:", len(rows))
print(rows[:3])

Rows: 50
[[11, 11000, 'random', 3, 0, 'hole'], [11, 11001, 'random', 6, 0, 'hole'], [11, 11002, 'random', 4, 0, 'hole']]


In [21]:
df = pd.DataFrame(
    rows,
    columns=[
        "batch_seed",
        "episode_seed",
        "policy",
        "steps",
        "total_reward",
        "outcome"
    ]
)

df.head()

,batch_seed,episode_seed,policy,steps,total_reward,outcome
0,11,11000,random,3,0,hole
1,11,11001,random,6,0,hole
2,11,11002,random,4,0,hole
3,11,11003,random,7,0,hole
4,11,11004,random,7,0,hole


In [22]:
goals = (df["outcome"] == "goal").sum()
holes = (df["outcome"] == "hole").sum()
time_limits = (df["outcome"] == "time_limit").sum()

success_pct = goals / 50 * 100
mean_steps = df["steps"].mean()

print("Goals /50:", goals)
print("Holes /50:", holes)
print("Time limits /50:", time_limits)
print("Success %:", round(success_pct, 2))
print("Mean steps:", round(mean_steps, 2))

Goals /50: 3
Holes /50: 47
Time limits /50: 0
Success %: 6.0
Mean steps: 7.14


In [23]:
summary_rows = []

for slippery in [False, True]:

    env = gym.make(
        "FrozenLake-v1",
        map_name="4x4",
        is_slippery=slippery
    )

    for policy in ["random", "fixed"]:

        for batch_seed in [11, 22, 33]:

            rows = []

            for i in range(50):

                episode_seed = 1000 * batch_seed + i

                state, total_reward, steps, outcome = run_episode(
                    env,
                    policy=policy,
                    seed=episode_seed
                )

                rows.append({
                    "batch_seed": batch_seed,
                    "episode_seed": episode_seed,
                    "policy": policy,
                    "steps": steps,
                    "total_reward": total_reward,
                    "outcome": outcome
                })

            df = pd.DataFrame(rows)

            goals = (df["outcome"] == "goal").sum()
            holes = (df["outcome"] == "hole").sum()
            time_limits = (df["outcome"] == "time_limit").sum()

            success_pct = goals / 50 * 100
            mean_steps = df["steps"].mean()

            summary_rows.append({
                "Slippery": slippery,
                "Policy": policy,
                "Batch seed": batch_seed,
                "Goals/50": goals,
                "Holes/50": holes,
                "Time limits/50": time_limits,
                "Success %": round(success_pct, 2),
                "Mean steps": round(mean_steps, 2)
            })

summary_df = pd.DataFrame(summary_rows)

summary_df

,Slippery,Policy,Batch seed,Goals/50,Holes/50,Time limits/50,Success %,Mean steps
0,False,random,11,0,50,0,0.0,6.84
1,False,random,22,0,50,0,0.0,8.80
2,False,random,33,2,48,0,4.0,8.64
3,False,fixed,11,50,0,0,100.0,6.00
4,False,fixed,22,50,0,0,100.0,6.00
5,False,fixed,33,50,0,0,100.0,6.00
6,True,random,11,1,49,0,2.0,8.04
7,True,random,22,0,50,0,0.0,6.98
8,True,random,33,0,50,0,0.0,7.04
9,True,fixed,11,3,47,0,6.0,8.76


In [24]:
summary_df.to_csv(
    "outputs/summary_results.csv",
    index=False
)

print("Saved.")

Saved.


In [25]:
with open("outputs/summary_results.txt", "w") as f:
    f.write(summary_df.to_string())

print("Saved.")

Saved.
